In [63]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

CSV_FILENAME = "datasets/titanic.csv"
df = pd.read_csv(CSV_FILENAME)
print(f"Veri başarıyla yüklendi. Boyut: {df.shape}")

Veri başarıyla yüklendi. Boyut: (891, 12)


In [57]:
# ==========================================
# 1. VERİ ANALİZİ
# ==========================================

print("\n" + "=" * 50)
print("🔍 EKSİK VERİ ANALİZİ (Missing Values)")
print("=" * 50)

# Sadece eksik verisi olan sütunları ve oranlarını hesapla
missing_count = df.isnull().sum()
missing_percent = 100 * df.isnull().mean()

missing_df = pd.DataFrame(
    {"Eksik Sayısı": missing_count, "Oran (%)": missing_percent}
)
# Sadece eksik değeri olanları filtrele ve büyükten küçüğe sırala
missing_df = missing_df[missing_df["Eksik Sayısı"] > 0].sort_values(
    by="Eksik Sayısı", ascending=False
)

if missing_df.empty:
  print("✨ Harika! Veri setinde hiç eksik değer yok.")
else:
  print(missing_df.to_string())

print("=" * 50 + "\n")


🔍 EKSİK VERİ ANALİZİ (Missing Values)
          Eksik Sayısı   Oran (%)
Cabin              687  77.104377
Age                177  19.865320
Embarked             2   0.224467



In [45]:
# ==========================================
# 1. MANUEL FEATURE ENGINEERING
# ==========================================
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
df["Title"] = df["Title"].replace(
    [
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona",
    ],
    "Rare",
)
df["Title"] = df["Title"].replace("Mlle", "Miss")
df["Title"] = df["Title"].replace("Ms", "Miss")
df["Title"] = df["Title"].replace("Mme", "Mrs")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = 0
df.loc[df["FamilySize"] == 1, "IsAlone"] = 1

df["Deck"] = df["Cabin"].fillna("U").astype(str).str[0]
df["Deck"] = df["Deck"].replace("T", "U")

features = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "FamilySize",
    "IsAlone",
    "Title",
    "Deck",
]
X = df[features].copy()
y = df["Survived"]

X["Age"] = X["Age"].fillna(X["Age"].median())
X["Fare"] = X["Fare"].fillna(X["Fare"].median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [49]:
# ==========================================
# 2. OTOMATİK FEATURE EXTRACTION & SELECTION İLE K-FOLD TESTİ
# ==========================================
numeric_cols = ["Age", "Fare", "FamilySize", "IsAlone", "Pclass"]
categorical_cols = ["Sex", "Title", "Deck"]

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "poly",
            PolynomialFeatures(
                degree=2, interaction_only=False, include_bias=False
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="passthrough",
)

pipeline_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=30)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 40)
print("--- K-Fold Cross Validation Sonuçları ---")
cv_scores = cross_val_score(pipeline_model, X, y, cv=skf, scoring="accuracy")

for i, score in enumerate(cv_scores, 1):
  print(f"Fold {i}: {score:.4f}")

print(f"\nGerçek K-Fold Başarısı (Ortalama): {np.mean(cv_scores):.4f}")
print(f"Skor Sapması (Standart Sapma)    : {np.std(cv_scores):.4f}")
print("=" * 40 + "\n")


--- K-Fold Cross Validation Sonuçları ---
Fold 1: 0.8380
Fold 2: 0.8371
Fold 3: 0.7697
Fold 4: 0.8315
Fold 5: 0.8539

Gerçek K-Fold Başarısı (Ortalama): 0.8260
Skor Sapması (Standart Sapma)    : 0.0292



In [50]:
# ==========================================
# ARA BÖLÜM: İDEAL K DEĞERİNİ BULMA
# ==========================================
for k_val in [10, 15, 20, 25, 30]:
    temp_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', SelectKBest(score_func=f_classif, k=k_val)),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])

    scores = cross_val_score(temp_pipeline, X, y, cv=skf, scoring='accuracy')
    print(f"k = {k_val} için Ortalama K-Fold Başarısı: {np.mean(scores):.4f}")

k = 10 için Ortalama K-Fold Başarısı: 0.7856
k = 15 için Ortalama K-Fold Başarısı: 0.7913
k = 20 için Ortalama K-Fold Başarısı: 0.8036
k = 25 için Ortalama K-Fold Başarısı: 0.8204
k = 30 için Ortalama K-Fold Başarısı: 0.8260


In [51]:
# ==========================================
# 3. NİHAİ MODEL EĞİTİMİ VE KAYIT
# ==========================================
pipeline_model.fit(X_train, y_train)

y_train_pred = pipeline_model.predict(X_train)
y_pred = pipeline_model.predict(X_test)

print(f"Train Doğruluk Oranı: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Model Doğruluk Oranı (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))



Train Doğruluk Oranı: 0.8385
Model Doğruluk Oranı (Accuracy): 0.8156

Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       105
           1       0.81      0.73      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.81       179

Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne kaydedildi!


In [61]:
# ==========================================
# 4. KARA KUTUYU AÇMA: Hangi Özellikler Seçildi?
# ==========================================
preprocessor = pipeline_model.named_steps["preprocessor"]
selector = pipeline_model.named_steps["feature_selection"]
classifier = pipeline_model.named_steps["classifier"]

all_feature_names = preprocessor.get_feature_names_out()
selected_mask = selector.get_support()
selected_features = all_feature_names[selected_mask]
coefficients = classifier.coef_[0]

feature_importance = pd.DataFrame(
    {"Özellik (Feature)": selected_features, "Etki Ağırlığı (Coefficient)": coefficients}
)

# --- GÖRSEL TEMİZLİK VE SIRALAMA BÖLÜMÜ ---
feature_importance["Özellik (Feature)"] = feature_importance[
    "Özellik (Feature)"
].str.replace(r"^(num__|cat__|text__|remainder__)", "", regex=True)

feature_importance = feature_importance.sort_values(
    by="Etki Ağırlığı (Coefficient)", key=abs, ascending=False
)

feature_importance["Etki Ağırlığı (Coefficient)"] = feature_importance[
    "Etki Ağırlığı (Coefficient)"
].round(4)
# ------------------------------------------

print("\n--- Modelin Seçtiği En İyi Özellikler ve Etkileri ---")
print(feature_importance.to_string(index=False))
print("Model Intercept (Sabit Terim):", classifier.intercept_)
print(
    "En düşük tahmin edilen olasılık:",
    pipeline_model.predict_proba(X_test)[:, 1].min(),
)


--- Modelin Seçtiği En İyi Özellikler ve Etkileri ---
 Özellik (Feature)  Etki Ağırlığı (Coefficient)
      Title_Master                       1.4896
         Title_Mrs                       1.0440
        Sex_female                       1.0319
            Deck_E                       1.0217
          Sex_male                      -1.0088
 FamilySize Pclass                      -0.9785
          Title_Mr                      -0.9593
            Deck_D                       0.7453
              Fare                       0.6839
            Pclass                      -0.6830
            Deck_U                      -0.5725
       Fare Pclass                       0.5655
   Fare FamilySize                      -0.3932
       Age IsAlone                       0.3838
            Deck_B                       0.3780
          Age Fare                       0.3301
               Age                      -0.3184
           IsAlone                       0.2978
        Title_Miss               

In [62]:
# ==========================================
# 5. MODELİ KAYDETME
# ==========================================
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(pipeline_model, "../backend/models/titanic_pipeline.pkl")
joblib.dump(list(X_train.columns), "../backend/models/model_columns.pkl")

print(
    "Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne"
    " kaydedildi!"
)

Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne kaydedildi!
